# Coding Agent Evaluation on Your Mac

In this notebook, you build and run an evaluation of a coding agent. The agent under test is Codex CLI on Amazon Bedrock. It does two kinds of tasks, both made from real pull requests: it reviews a change and reports the defects, and it fixes a bug from an issue report. Hidden tests and LLM judges grade each result.

Each trial runs on your Mac, in a macOS sandbox, with its own sealed copy of the repository. The agent can reach Amazon Bedrock and nothing else. It cannot reach GitHub, where the answers are, and it cannot read the hidden tests.

| | Approximately |
|---|---|
| **Time** | 40 minutes of reading, and 12 minutes of run time for the default path. The optional set of 12 trials in section 8 adds 38 minutes |
| **Amazon Bedrock cost** | Under $0.50 for the default path. About $3 for the set of 12 trials |
| **What is created** | On your Mac: a toolchain in this folder (about 1.8 GB), sealed clones and trial folders in `~/.coding-assistant-local/<id>/work` (`<id>` is a short hash of this folder's path, so each checkout has its own), and the trial records in `jobs/`. In AWS: no resources. The runner makes temporary credentials, which expire on their own, and Bedrock calls. Section 9 removes everything |

> **Start Jupyter from a plain terminal.** The agent runs inside a macOS sandbox (`sandbox-exec`, through `srt`). macOS does not let a process that is already inside a sandbox start another one. So start Jupyter from Terminal.app or iTerm, not from a terminal inside an agent tool, an IDE sandbox or another sandbox. The check in section 2 finds this and tells you what to do.
>
> From the folder of this notebook (the one with `runner/p1.py`):
>
> ```bash
> UV_CACHE_DIR=uv-cache UV_PYTHON_INSTALL_DIR=uv-python JUPYTER_DATA_DIR=home/jupyter uv run --no-project --python 3.12 --with jupyterlab jupyter lab "05 Local mode walkthrough.ipynb"
> ```
>
> The variables keep Jupyter's own packages and settings in this folder, with the rest of the toolchain, so section 9 removes them too. `JUPYTER_DATA_DIR` also makes sure that this notebook runs on the Python of this command, not on a `python3` kernel that you registered for another project.

### Procedure

1. What you will build and learn (about 25 minutes of reading).
2. Check the prerequisites.
3. Set up the toolchain and the sealed clones.
4. Look at the datasets, the tasks and the graders.
5. See what the sandbox lets the agent reach.
6. Run one PR-review trial and one coding trial, and read the results.
7. Run the controls: the empty agent scores 0, the oracle scores 1.
8. The set of 12 trials, and how to read it.
9. Clean up.

## 1. What You Will Build and Learn

### What is an eval for a coding agent?

An eval is a set of examples where you already know the correct answer. You run the agent on each example and compare its output to the answer. The result is a score that you can track over time. If you are new to evals, start with the [Foundational Evaluations](https://github.com/aws-samples/sample-gen-ai-evaluations-workshop/tree/main/Foundational%20Evaluations) modules.

For a chatbot, an example is a prompt and an expected response. For a coding agent, an example can be much larger. It contains a full repository, a task in that repository, and the correct result.

This notebook evaluates two types of coding agents:

- A **PR review agent** reads a change and reports the defects. Then it approves the change or requests changes.
- A **coding agent** reads an issue and writes a patch that fixes it.

### The problem

Two problems make coding evals more difficult than other evals.

**The examples should come from your own code.** Public benchmarks, for example SWE-bench and [Code Review Bench](https://github.com/withmartian/code-review-benchmark), use PRs from popular open-source projects. They test for general defects, for example race conditions and missing null checks. A high score shows that the agent can find general defects. It does not show that the agent knows the rules of your team.

Here is an example. A coding agent opens a PR that moves a plan ID from Stripe metadata into your database. The agent does not know that a different billing service reads that metadata at each renewal. The tests for each service pass. One month later, the renewals reset customers to their old plans. A generic review agent approves this PR, because the change looks like a good cleanup. A reviewer who knows your system blocks it. To measure this skill, you need examples from your own code.

**Each example needs a full, sealed environment.** A review agent cannot review a diff alone. It must read the code around the change, and it must sometimes run the tests. Thus each example needs the repository at the correct commit, with all dependencies installed. The environment must also hide the answer. The correct fix is in a later commit, and the review comments are on GitHub. Agents search for data when they can. If the agent finds the answer, the score measures search, not review.

### The approach: use your git history as the dataset

You already have a labeled dataset. Each PR in your repository records the code before the change, the change, what the reviewer said, and the fix that the reviewer approved. Your team did the labeling work during normal code review.

| Item | Where git keeps it | Use in the eval |
|------|--------------------|-----------------|
| The code before the change | The parent commit | The start state for the agent |
| The change | The diff from the parent commit to the PR commit | The input for a PR review agent |
| The reviewer decision | The PR review comments and the verdict | The answer for a PR review agent |
| The correct fix | The merged patch and its tests | The answer for a coding agent |

Each example is a commit hash and a PR number, so you can make the same example again at any time. Keep the finished examples in a git repository too. Then each version of your eval has a commit, and you can follow scores across versions.

For each PR, the approach has three steps:

1. **Roll back the repository to the parent commit of the change.** The agent sees the code as the reviewer saw it, before the fix. If you start after the fix, the defect is not there.
2. **Seal the environment.** Remove all paths to the answer. Then the score measures the skill of the agent, not its ability to search.
3. **Use the PR review or the merged patch as the oracle.** The oracle is the gold standard answer. The grader compares the agent output to the oracle.

```
parent commit ──► PR change ──► review ──────────► fix ──► merge
      ▲               │            │                         │
  roll back here    input      oracle for a            oracle for a
                               PR review agent         coding agent
```

### How each trial is sealed

A seal has three layers. Each layer closes one path to the answer.

| Layer | Path to the answer | Control in this notebook |
|-------|--------------------|--------------------------|
| **Git** | `git log --all` shows the fix commit | `p1 prepare` clones only the parent commit, removes the remote and the remote refs, expires the reflog and prunes. Each trial gets a copy of that clone |
| **Network** | `gh pr view` shows the review comments, and GitHub has the fix | The `srt` sandbox lets the agent reach two Amazon Bedrock hosts on port 443 and nothing else: no GitHub, no package index |
| **Verifier** | The agent reads or changes the hidden tests or the oracle | `tests/` and `solution/` stay in the module folder, which the agent's sandbox cannot read. The verifier runs after the agent stops, in a second sandbox. A coding verifier first restores the base test files, so the agent cannot change a test so that it passes |

Why this approach solves the problem:

| Problem | Solution |
|---------|----------|
| Public benchmarks do not know your code. | The examples come from your PR history. They contain the rules of your team. |
| An environment for each example takes too much manual work. | Each task lists the steps that build its start state. The runner does these steps for each trial. |
| The agent can find the answer. | The seal closes the git, network and verifier paths. |

### The task format

Each example is in a standard format. Then a tool can run any agent on any example, and other people can run your eval again and get the same result. The format of this module is one **task directory** for each example. Many benchmarks use the same layout, for example [Senior SWE-Bench](https://senior-swe-bench.snorkel.ai/blog/2026-06-16-how-it-works).

```text
tasks/<task>/
├── task.toml            # the settings of the agent phase and the verifier phase, and the metadata
├── instruction.md       # what the agent reads
├── environment/
│   ├── Dockerfile       # the build recipe of the start state: clone, seal, install the dependencies
│   ├── change.patch     # PR review only: the change under review
│   └── requirements.lock  # coding only: the pinned dependencies
├── tests/               # verifier only
│   ├── test.sh          # the verifier entry point. It writes the reward file
│   └── ...              # the grader, the oracle or the hidden tests, the judge settings
└── solution/            # verifier only: the reference answer, to test the grader
```

The most important part of this format is the split between the agent and the verifier:

- **The agent sees** `instruction.md` and the start state that `environment/` describes. It does not see `task.toml`.
- **Only the verifier sees** `tests/` and `solution/`.

The environment file is a `Dockerfile`: a list of build steps, one per line, in a syntax that many tools read. In this notebook nothing builds it as an image. The runner reads its steps and does them on your Mac (section 4).

The task files use fixed paths: `/work/repo` for the repository, `/tests` for the verifier files and `/logs` for the outputs. The runner replaces them with the paths of each trial.

`task.toml` has these parts:

| Table | Fields | What they do |
|---|---|---|
| `[task]` | `name`, `description` | The name in the trial records |
| `[agent]` | `timeout_sec` | How long the agent can run |
| `[environment]` | `network_mode = "allowlist"`, `allowed_hosts` | The network of the agent phase: only the Bedrock hosts. Never `github.com` |
| `[verifier]` | `timeout_sec`, `environment_mode`, and for a coding task `network_mode` and `allowed_hosts` | `separate`: the verifier has its own environment (PR review: it only reads the findings). `shared`: the verifier runs in the agent's environment, because it must run the agent's code (coding) |
| `[verifier.env]` | `JUDGE_MODEL_ID` or `VERIFIER_MODEL_ID`, the `AWS_*` values | The judge model and the credentials of the verifier. The agent phase never gets them |
| `[metadata.*]` | `origin`, and `harvest` (PR review) or `verifier` and `narrative` (coding) | Where the datapoint came from, the expected verdict, the hidden tests, the cause and the fix. Not visible to the agent |

### How a trial runs

A **trial** is one agent on one task. The runner, `runner/p1.py`, does each trial in a fresh trial folder, and deletes the folder after it records the result. Thus each trial starts from the same state, and trials cannot change each other.

```
 Jupyter (started from a plain terminal)
   |
   |  setup.sh        Node 22, srt, Codex, ripgrep  ->  this folder only
   |  p1 prepare      git clone + seal, once per task  ->  ~/.coding-assistant-local/<id>/work/base/<task>/
   v
 p1 trial  --  aws sts: agent token (1 h, 3 model IDs), verifier token (2 h)
   |
   |  copy of the sealed clone + uv venv  ->  ~/.coding-assistant-local/<id>/work/trials/<trial>/
   v
 +----------------------------- srt sandbox (agent profile) ------------------------------+
 |  Codex CLI   reads and writes only its trial folder                                    |
 |              reaches only bedrock-mantle and bedrock-runtime (us-east-1, port 443)     |
 +----------------------------------------------------------------------------------------+
   |  the agent stops
   v
 +----------------------------- srt sandbox (verifier profile) ---------------------------+
 |  tests/test.sh: hidden tests, base suite, validation agent, rubric and taste judges    |
 |              reaches only bedrock-runtime                                              |
 +----------------------------------------------------------------------------------------+
   |
   v
 jobs/<job>/<trial>/   result.json, verifier/reward.json, agent/codex.txt, agent/sessions/
```

The work folder `~/.coding-assistant-local/<id>/work` is outside the module folder on purpose: the module path has spaces ("Workload Specific Evaluations"), and the verifier scripts use paths without quotes.

Before you can trust a score, each task must build and run correctly. Do these checks for each task, in this order:

1. **Prepare the start state.** The clone must be at the parent commit and sealed, and the dependencies must install. A failure here costs no LLM tokens.
2. **Run the controls.** The oracle agent submits the reference answer: the score must be perfect. The empty agent changes nothing: the score must be 0. If not, the grader has a defect.
3. **Run the agent.** Only now does the score tell you about the agent.

### The models

All model IDs of the module are in `model_config.py`:

| Name | Model | Role |
|---|---|---|
| `CODEX_MODEL_ID` | `openai.gpt-6-sol` | The agent under test. Codex calls it through the Bedrock Mantle endpoint (`bedrock-mantle.us-east-1.api.aws`) |
| `JUDGE_MODEL_ID` | `us.anthropic.claude-haiku-5-5` | The PR review judge: is an agent finding the same defect as an oracle finding? |
| `VERIFIER_MODEL_ID` | `us.anthropic.claude-haiku-5-5` | The coding verifier: the validation agent, the validation judge, the rubric judge and the taste judge |

The judges use a different model from the agent, from a different model family. A judge can prefer answers that read like its own, so a judge that is the same model as the agent can give that agent too much credit. The judge model is also small and fast, because a coding trial makes many judge calls. The verifier records the model that ran in each trial, so you can see it in the results.

### The agent: Codex CLI on Amazon Bedrock

[Codex CLI](https://github.com/openai/codex) is an open-source coding agent that runs in a terminal. It reads the repository, runs shell commands and edits files in a loop until it decides that the task is done. `codex exec` runs it without a person: it gets the instruction as its prompt, and it writes each step as a JSON event.

Codex reaches Amazon Bedrock with the settings in `codex-bedrock.toml`: the provider `amazon-bedrock`, the Region `us-east-1`, and no update check, analytics or feedback calls. It needs no API key. The runner copies this file to the trial's Codex home as `config.toml`, and gives Codex the AWS credentials in its environment only. Your own `~/.codex/config.toml` is not used.

The runner starts Codex with these flags (`CODEX_FLAGS` in `runner/p1.py`; section 4 prints them):

| Flag | Purpose |
|---|---|
| `--model CODEX_MODEL_ID` | The model on Bedrock |
| `--dangerously-bypass-approvals-and-sandbox` | Codex does not stop to ask for approval, and it does not start its own sandbox. The `srt` sandbox around it is the seal, and macOS does not let a sandbox start inside another one |
| `--skip-git-repo-check` | Codex runs in the trial's clone without its own git checks |
| `--json` | Write each step as a JSON event to `agent/codex.txt` |
| `--enable unified_exec` | A Codex feature setting for how it runs shell commands |
| `-c web_search=disabled` | No web search. The sandbox blocks it too |

**Credentials.** For each trial, the runner makes two short-lived credentials with `aws sts`: a 1-hour federated token for the agent, which `scripts/agent-policy.json` limits to Bedrock inference (the Mantle endpoint and three models), and a 2-hour session token for the verifier. They are passed only in the environment of the processes that need them, and never written to disk. The agent can use its token for every Bedrock call that the policy allows, so the policy lists only the models.

### What was measured

On one Mac (macOS on Apple silicon, November 2026, n=3 per task), 12 Codex trials ran with 0 invalid trials, in 37.7 minutes. The coding tasks got reward 1 in 6 of 6 trials. The review passed on the approve task in 3 of 3 trials and failed on the reject task in 3 of 3. The sandbox held against scripted probes and two LLM red-team runs. This is a preview: it is tested on macOS on Apple silicon only. [README.md](README.md) has the details.

## 2. Check the Prerequisites

| Need | Why |
|---|---|
| macOS on Apple silicon | `setup.sh` downloads arm64 builds, and `srt` uses `sandbox-exec` (Seatbelt) |
| Jupyter started from a plain terminal | a sandbox cannot start inside another sandbox |
| The checkout under your home folder | the sandbox denies reads of your home folder. A checkout somewhere else (for example `/opt`) would be readable by the agent, including the hidden tests |
| `uv`, `git`, `curl`, `tar`, AWS CLI v2 | `brew install uv`; the `git`, `curl` and `tar` of macOS are fine. `env.sh` looks for tools in `/opt/homebrew/bin`, `/usr/local/bin` and `/usr/bin` only |
| An IAM user access key in the default AWS profile | the runner calls `aws sts get-session-token` and `get-federation-token`, which AWS does not allow with role or IAM Identity Center credentials. TESTER-GUIDE.md section 2 has a minimal policy |
| Model access in `us-east-1` | the agent model and the judge model in `model_config.py` |
| About 3 GB of free disk | toolchain, uv cache, sealed clones and trial folders |

You do not need `gh`, a system Python or `sudo`.

This first cell sets the paths and one helper, `local()`. It runs a command in this folder in the same way as a new plain terminal after `source env.sh`: your shell's `AWS_*` variables, the Jupyter environment and its virtual environment are not passed on. The runner's output comes back through a pipe and is printed here. (Do not redirect the runner's output to a file yourself: inside the sandbox, `node` aborts when its output is a file in your home folder.)

In [ ]:
import hashlib
import importlib.util
import json
import os
import platform
import re
import shlex
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import tomllib
from datetime import datetime
from pathlib import Path

from IPython.display import Markdown, display

DRY_RUN = os.environ.get("LOCAL_NB_DRY_RUN") == "1"  # maintainers: print the commands, do not run them

LOCAL = Path.cwd()       # this notebook's folder
MODULE = LOCAL           # datasets/, model_config.py and codex-bedrock.toml are in this folder too
assert (LOCAL / "runner" / "p1.py").exists(), "Open this notebook from its own folder (the one with runner/p1.py)."
REAL_HOME = os.environ.get("P1_REAL_HOME") or str(Path.home())        # env.sh changes HOME; it keeps the real one here
WORK_ID = hashlib.sha1(str(LOCAL.resolve()).encode()).hexdigest()[:8]   # one work folder per checkout of this folder
os.environ.setdefault("P1_WORK", f"{REAL_HOME}/.coding-assistant-local/{WORK_ID}/work")  # the same default as env.sh
P1_WORK = Path(os.environ["P1_WORK"])

sys.dont_write_bytecode = True                       # no __pycache__ in the module folder
sys.path[:0] = [str(MODULE), str(LOCAL / "runner")]
from model_config import CODEX_MODEL_ID, JUDGE_MODEL_ID, VERIFIER_MODEL_ID
import p1 as runner                                  # the local runner, imported to read its settings


def mask(text: str) -> str:
    """Hide your home path and any 12-digit number (an AWS account ID) in what this notebook prints."""
    return re.sub(r"(?<!\d)\d{12}(?!\d)", "<12 digits>", text.replace(REAL_HOME, "~"))


read_json = lambda path: json.loads(Path(path).read_text())
read_toml = lambda path: tomllib.loads(Path(path).read_text())


_tool_dirs = [str(Path(p).parent) for p in map(shutil.which, ("uv", "aws", "git")) if p]
SHELL_ENV = {"PATH": ":".join(dict.fromkeys(_tool_dirs + ["/opt/homebrew/bin", "/usr/local/bin", "/usr/bin", "/bin", "/usr/sbin", "/sbin"])),
             "HOME": REAL_HOME, "LANG": "en_US.UTF-8", "TERM": "dumb",
             "USER": os.environ.get("USER", ""), "TMPDIR": os.environ.get("TMPDIR", "/tmp")}


def local(cmd: str, check: bool = True, capture: bool = False) -> str | None:
    """Run `cmd` in this folder after `source env.sh`, as in a new plain terminal. Print the output as it arrives.
    Return the output if `capture` is True."""
    print(f"$ {cmd}", flush=True)
    if DRY_RUN:
        return "" if capture else None
    t0 = time.time()
    proc = subprocess.Popen(["/bin/bash", "-c", f"source ./env.sh && {cmd}"], cwd=LOCAL, env=SHELL_ENV, text=True,
                            stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
    lines = []
    try:
        for line in proc.stdout:
            lines.append(line)
            print(mask(line), end="", flush=True)
        rc = proc.wait()
    except KeyboardInterrupt:  # "Interrupt the kernel" stops the runner like Ctrl-C, so it releases its lock
        os.killpg(proc.pid, signal.SIGINT)
        proc.wait()
        raise
    print(f"[exit {rc} after {time.time() - t0:.0f} s]")
    if check and rc != 0:
        raise RuntimeError(f"The command failed (exit {rc}). Read the lines above. TESTER-GUIDE.md section 9 lists the usual causes.")
    return "".join(lines) if capture else None


P1 = "uv run --no-project --python 3.12 runner/p1.py"   # the guide's `p1` shorthand
FED = "--federated scripts/agent-policy.json"            # the agent gets a 1-hour token limited to three model IDs
TASK_NAMES = ["pr371-fix", "pr338-fix", "pr4121-approve", "pr4615-reject"]
print("Module:", MODULE.name, "| work folder:", mask(str(P1_WORK)), "| dry run" if DRY_RUN else "")

### Do the setup checks

Each check prints `OK` or `ERR` and what to do. Run this cell again after you repair an item.

In [ ]:
def run(cmd: list[str], env: dict | None = None, timeout: int = 60) -> tuple[bool, str]:
    """Run a command. Return (ok, output)."""
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, stdin=subprocess.DEVNULL, env=env or SHELL_ENV)
        return out.returncode == 0, (out.stdout + out.stderr).strip()
    except (FileNotFoundError, subprocess.TimeoutExpired) as error:
        return False, str(error)


ENV_SH_DIRS = ("/opt/homebrew/bin", "/usr/local/bin", "/usr/bin")   # env.sh sets PATH to these (and the toolchain)


def check_platform():
    ok = platform.system() == "Darwin" and platform.machine() == "arm64"
    return ok, f"{platform.system()} {platform.machine()} {platform.mac_ver()[0]}" + ("" if ok else ": this notebook needs macOS on Apple silicon")


def check_python():
    ok = sys.version_info >= (3, 11)
    return ok, f"kernel Python {sys.version.split()[0]}" + ("" if ok else ": start Jupyter with --python 3.12 (first cell)")


def check_under_home():
    try:
        LOCAL.resolve().relative_to(Path(REAL_HOME).resolve())
        return True, "the checkout is under your home folder"
    except ValueError:
        return False, ("Move the checkout under your home folder. The sandbox hides your home folder from the agent; "
                       "outside it, the agent could read the hidden tests.")


def check_work_dir():
    ok = " " not in str(P1_WORK)
    return ok, mask(str(P1_WORK)) if ok else "Your home path has a space. Set P1_WORK to a path without spaces before you start Jupyter."


def check_tool(name: str, args: list[str], hint: str):
    path = shutil.which(name, path=SHELL_ENV["PATH"])
    if not path:
        return False, f"{name} is not on PATH. {hint}"
    if str(Path(path).parent) not in ENV_SH_DIRS:
        return False, f"{name} is in {mask(str(Path(path).parent))}, which env.sh does not search. {hint}"
    ok, out = run([path, *args])
    return ok, (out.splitlines() or [""])[0]


def check_aws_v2():
    ok, detail = check_tool("aws", ["--version"], "Install AWS CLI v2: https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html")
    return ok and detail.startswith("aws-cli/2"), detail if detail.startswith("aws-cli/2") or not ok else f"{detail}: the runner needs AWS CLI v2"


def check_sandbox():
    ok, out = run(["/usr/bin/sandbox-exec", "-p", "(version 1)(allow default)", "/usr/bin/true"])
    if ok:
        return True, "a sandbox can start from this kernel"
    return False, (f"a sandbox cannot start here ({out[-100:]}). Jupyter runs inside another sandbox. Stop it, "
                   "open Terminal.app or iTerm, start Jupyter there with the command in the first cell, and run this cell again.")


checks = {
    "macOS, Apple silicon": check_platform(),
    "Python 3.11+":         check_python(),
    "plain terminal":       check_sandbox(),
    "checkout under ~":     check_under_home(),
    "work folder":          check_work_dir(),
    "uv":                   check_tool("uv", ["--version"], "Install it with `brew install uv`."),
    "git":                  check_tool("git", ["--version"], "Install the Xcode command line tools: `xcode-select --install`."),
    "curl":                 check_tool("curl", ["--version"], "curl ships with macOS."),
    "tar":                  check_tool("tar", ["--version"], "tar ships with macOS."),
    "AWS CLI v2":           check_aws_v2(),
}
for name, (ok, detail) in checks.items():
    print(f"{'OK ' if ok else 'ERR'} {name:<22} {detail}")
failed = [name for name, (ok, _) in checks.items() if not ok]
if failed and not DRY_RUN:
    raise RuntimeError(f"Repair these items, then run this cell again: {', '.join(failed)}")
print("\nAll checks passed." if not failed else "\nDry run: checks not enforced.")

### Check your AWS identity and Bedrock access

The runner reads only the **default** AWS profile (it ignores `AWS_PROFILE` and the `AWS_*` variables), so this check does the same. It prints the type of the identity, not its name or account. Then it makes one short call to each judge model. Codex's model is checked from inside the sandbox in section 5 (stage L6a).

In [ ]:
if DRY_RUN:
    print("Dry run: no AWS calls.")
else:
    ok, out = run(["aws", "sts", "get-caller-identity", "--output", "json"])
    if not ok:
        raise RuntimeError("The default AWS profile has no working credentials: " + mask(out[-300:]))
    kind = json.loads(out)["Arn"].split(":")[5].split("/")[0]  # user, assumed-role, federated-user or root
    print("Default profile identity type:", kind)
    if kind != "user":
        raise RuntimeError("The runner needs an IAM user access key in the default profile. Role and IAM Identity Center "
                           "credentials cannot call get-federation-token (TESTER-GUIDE.md section 2).")

    message = json.dumps([{"role": "user", "content": [{"text": "Reply with the single word: yes"}]}])
    for model in sorted({JUDGE_MODEL_ID, VERIFIER_MODEL_ID}):
        ok, out = run(["aws", "bedrock-runtime", "converse", "--region", runner.REGION, "--model-id", model,
                       "--messages", message, "--inference-config", '{"maxTokens": 256}',
                       "--query", "output.message.content[0].text", "--output", "text"])
        if not ok:
            raise RuntimeError(f"Cannot call {model} in {runner.REGION}: {mask(out[-300:])}\n"
                               "Request access to the model in the Amazon Bedrock console.")
        print(f"OK  {model} replied: {out.strip()[:40]}")

## 3. Set Up the Toolchain

`setup.sh` downloads Node 22, `srt` (the sandbox), Codex CLI and ripgrep, with pinned versions, and writes a small `timeout` shim (macOS has no GNU `timeout`, and the verifier calls it). Everything goes into this folder:

| Folder | Content | Size |
|---|---|---|
| `node/`, `npm/`, `npm-cache/` | Node 22, `srt`, Codex CLI | about 0.7 GB |
| `bin/` | `rg`, `timeout` | 4 MB |
| `uv-cache/`, `uv-python/` | Python 3.12 and the task packages, filled by the first environment build | about 1.1 GB |
| `home/` | the `HOME` of the toolchain (`env.sh` sets it) | under 1 MB |

Nothing is installed system-wide, nothing needs `sudo`, and git ignores all of it. Measured: 36 s the first time, 1 s when you run it again.

In [ ]:
local("./setup.sh")

### Prepare the sealed clones

`p1 prepare` does the git steps of each task's environment file on your Mac: clone the repository at the base commit, remove the remote and the remote refs, expire the reflog and prune everything that is not reachable. After this, the clone has one commit. For a PR-review task it then applies the change under review as uncommitted edits. Each trial later gets a copy-on-write copy of this clone, so the agent always starts from the same state.

Measured: about 30 s for the four tasks the first time. A task that is already prepared is skipped (0 s).

In [ ]:
local(f"{P1} prepare {' '.join(TASK_NAMES)}")

## 4. Look at the Datasets, the Tasks and the Graders

The module has two datasets in `datasets/`, with two tasks each:

| Dataset | Tasks | Agent type |
|---|---|---|
| `awslabs-mcp-pr-review` | `pr4615-reject`, `pr4121-approve` | PR review, from [awslabs/mcp](https://github.com/awslabs/mcp) |
| `strands-evals-coding` | `pr371-fix`, `pr338-fix` | Coding, from [strands-agents/evals](https://github.com/strands-agents/evals) |

We made the datapoints before the workshop. It is not necessary to make datapoints in this notebook. The runner's `Task` class reads a task directory: where the repository is, its base commit, and if it is a coding task or a PR-review task.

In [ ]:
TASKS = {name: runner.Task(name) for name in TASK_NAMES}
for name, t in TASKS.items():
    print(f"{name:<15} {t.kind:<6} {t.url}  base {t.sha[:9]}  change under review: {t.has_change}")

### How the datapoints were made

We used [sample-code-eval-harvester](https://github.com/aws-samples/sample-code-eval-harvester) to make the PR review datapoints. It is a CLI named `eval-harvest`. A coding agent runs the CLI: the CLI collects the facts from git and GitHub, and the agent makes the decisions.

| Step | Command | Who does it | Result |
|------|---------|-------------|--------|
| 1 | `init` | CLI | A dataset directory. A rubric and a risk map, from the conventions of the repository |
| 2 | `survey` | CLI | A list of the PRs that can become datapoints, and the reason that the other PRs cannot |
| 3 | `capture` | CLI | A candidate file for each PR: the review iterations, the patches, the review comments and the reviews |
| 4 | `brief`, `show`, `annotate` | Agent | A label for each review comment. A finding with a severity for each defect. A risk level for the change |
| 5 | `emit` | CLI | A task directory with a sealed environment |
| 6 | `verify` | CLI | Checks that the datapoint is correct, and that the agent cannot see the answer |

In step 4, the agent gives one of these labels to each review comment. Only defects go into the oracle.

| Label | Meaning | In the oracle |
|-------|---------|---------------|
| **Defect** | The reviewer blocks the PR for this problem | Yes, with severity `low`, `medium` or `high` |
| **Nit** | Style or preference | No |
| **Question** | A request for information | No |
| **Approval** | A sign-off with no finding | No |
| **Bot** | A comment from an automated tool | No |

> **Note:** Do not label a nit as a defect. If you do, the eval rewards an agent that reports nits.

The coding datapoints were made by hand, in these steps:

1. **Select merged PRs that fix a bug.** Each PR must close an issue that describes the problem, and its tests must run without a network or a model. For example, a test that calls Amazon Bedrock cannot be a hidden test.
2. **Write the instruction from the issue.** Keep the symptom and the expected behavior. Remove the root cause, the file names and the function names. The agent must find the cause itself.
3. **Write the hidden tests.** Do not copy the PR tests. The PR tests often import private functions, or check one design of the fix. Write tests that use only the public API, and that check only the behavior that the instruction asks for. A different correct fix must pass.
4. **Write the validation spec and the rubric.** The spec has user stories, a procedure for each story, and several cases (inputs and expected values). The rubric has fail-to-pass and pass-to-pass criteria.
5. **Check every mechanism on the oracle and on the empty agent.** All fail-to-pass tests fail on the base, and all pass-to-pass tests pass. The oracle passes every test and every validation story, and it is a tasteful solve. The empty agent fails. Section 7 runs these controls.
6. **Write the environment file.** Clone only the base commit, and install the dependencies. The agent cannot find the fix in the git history, and it can run the tests.

To select the PRs and make the datapoints takes time and judgment. You can use the same tool and the same steps on your own repository.

### The task directories

These are the files of one task of each kind. The verifier files that are the same in each task of a dataset have one source in `datasets/_shared/`. Each task keeps a full copy, so that a task directory is complete by itself. `datasets/_shared/sync.py` copies the files into each task. The cell also checks that the copies match the source.

A PR-review task also has `tests/Dockerfile`: the recipe of the verifier's own environment, Python 3.12 with `boto3`. The runner makes the same environment once, with `uv`, in the work folder.

In [ ]:
def show_tree(root: Path) -> None:
    """Print the files of a task directory as a tree."""
    print(root.name + "/")
    for path in sorted(root.rglob("*")):
        if "__pycache__" not in path.parts:
            print("    " * len(path.relative_to(root).parts) + path.name + ("/" if path.is_dir() else ""))


for name in ("pr4615-reject", "pr371-fix"):
    show_tree(TASKS[name].dir)
    print()

spec = importlib.util.spec_from_file_location("shared_sync", MODULE / "datasets" / "_shared" / "sync.py")
sync = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sync)
stale = sync.stale_copies()
print(f"Shared verifier files: {len(sync.pairs())} copies, " +
      ("all match datasets/_shared/." if not stale else f"{len(stale)} differ. Run `python datasets/_shared/sync.py` in this folder."))

### `task.toml`

Make sure that:

- The agent `network_mode` is `allowlist`, and `allowed_hosts` contains only Bedrock hosts. The agent cannot reach GitHub, where the fix and the review are, or a package index.
- The PR-review verifier is `separate`: it reads only the findings file, so the agent cannot reach the oracle. The coding verifier is `shared`, because it must run the agent's code, and its own allowlist has only the Bedrock runtime host, for the validation agent and the judges.
- The verifier gets the judge model and the AWS credentials in `[verifier.env]`. The agent phase does not get them.

The runner gives `srt` the same hosts, with the port. It gives both verifiers only the Bedrock runtime host, which is stricter than the `public` network of the PR-review verifier in its `task.toml`.

In [ ]:
for name, task in TASKS.items():
    cfg = read_toml(task.dir / "task.toml")
    env, verifier = cfg["environment"], cfg["verifier"]
    verifier_network = verifier.get("network_mode") or verifier.get("environment", {}).get("network_mode")
    print(name)
    print(f"  agent:        {env['network_mode']} {env['allowed_hosts']}, timeout {cfg['agent']['timeout_sec']:.0f} s")
    print(f"  verifier:     {verifier.get('environment_mode', 'shared')}, network {verifier_network} {verifier.get('allowed_hosts', '')}, "
          f"timeout {verifier['timeout_sec']:.0f} s")
    print(f"  verifier env: {sorted(verifier.get('env', {}))}")
    assert env["network_mode"] == "allowlist", "The agent network must be an allowlist."
    assert all("bedrock" in host for host in env["allowed_hosts"]), "Allow only the Bedrock hosts."
print("\nrunner, agent:   ", runner.AGENT_HOSTS)
print("runner, verifier:", runner.VERIFIER_HOSTS)

### `instruction.md`

For a PR-review task, the instruction must be the same for all tasks. It must not refer to the PR, the reviewer or the findings. If it does, the agent can learn the answer from the instruction. It tells the agent to write its findings as JSON to the artifacts folder, where the verifier reads them.

For a coding task, the instruction is an issue report. It tells the symptom and the expected behavior. It does not name the file, the function or the cause. The last part, *General instructions*, is the same for all coding tasks.

The runner replaces the fixed paths with the trial's paths, and the sentence about where the agent works with "You are working on a laptop, in a sandbox with its own copy of the repository." Nothing else changes. Each trial record keeps the exact change in `agent/instruction.diff`. This cell prints what the agent reads.

In [ ]:
texts = {(TASKS[n].dir / "instruction.md").read_text() for n in ("pr4121-approve", "pr4615-reject")}
assert len(texts) == 1, "The PR-review instructions are not the same for all tasks."
for name in ("pr4615-reject", "pr371-fix"):
    rendered, _ = runner.render_instruction(TASKS[name], Path("<trial folder>"))
    print(f"==== {name}\n{rendered}\n")

### The environment file (a Dockerfile)

`environment/Dockerfile` is the build recipe of the agent's start state. The runner reads it and does its steps on your Mac. Make sure that the recipe does these steps:

| Step | Why | Where the runner does it |
|---|---|---|
| 1. Clone only the base commit | The fix is not in the clone | `p1 prepare` |
| 2. Remove the remote and the remote refs, expire the reflog, run garbage collection | Only one commit is left in the repository | `p1 prepare` |
| 3. PR review: apply `change.patch` as uncommitted edits | The agent reviews these changes with `git diff` | `p1 prepare` |
| 4. Install the locked dependencies | The agent can run the tests and try inputs, as a human reviewer can. A coding task pins each package in `requirements.lock` to the newest version on the day of the fix, so the datapoint does not change over time | a fresh `uv` environment per trial |
| 5. Put the Python environment on `PATH` for login shells | Codex runs each command in a login shell | the trial's own home folder |
| 6. Run the full test suite once | If a test fails before the agent starts, the datapoint is unfair: a failure in a trial must come from the agent | `p1 base-suite` (stage L1 of `run_all.sh`) |
| 7. Install the agent | The agent has no network for packages at run time | `setup.sh`, Codex 0.162.1 |

The environment has a network while it is built, but the agent has only its allowlist. Thus all the software must be installed first: the agent cannot install packages when it runs. This cell prints the git and dependency steps of one task of each kind.

In [ ]:
for name in ("pr4615-reject", "pr371-fix"):
    recipe = (TASKS[name].dir / "environment" / "Dockerfile").read_text()
    print(f"== {name}")
    for line in recipe.splitlines():
        if ("git " in line or "uv " in line) and not line.lstrip().startswith("#"):
            print("  " + line.strip()[:150])

### The PR review datapoints

The dataset has a reject datapoint and an approve datapoint. Both changes are security-relevant. One changes a guard against SQL that mutates a database. The other changes TLS certificate verification.

| | Reject datapoint | Approve datapoint |
|---|---|---|
| PR | [#4615](https://github.com/awslabs/mcp/pull/4615) | [#4121](https://github.com/awslabs/mcp/pull/4121) |
| Server | `mysql-mcp-server` | `eks-mcp-server` |
| Change | Read-only mode also rejects SQL statements that start with `DO`, `START`, `SHUTDOWN` and other mutating verbs | The Kubernetes client accepts EKS cluster CAs that have no Authority Key Identifier |
| Version under review | Iteration 0. A reviewer requested changes on this version | Iteration 1. This is the final version. Two reviewers approved it |
| Expected verdict | `block` | `approve` |
| Oracle | 1 high, 2 medium and 3 low findings | No findings |
| Risk (structural, classified) | high, high | medium, high |

**PR 4615 (reject).** The PR tries to close a bypass of the read-only guard. A reviewer found that the bypass is still open: a MySQL comment such as `#x` before the verb hides the verb from the new check. This is a clear high-severity defect. The reviewer also gave a test that shows the defect. A good review agent must find it.

**PR 4121 (approve).** EKS makes cluster CAs without an Authority Key Identifier. OpenSSL 3 rejects these CAs when the `VERIFY_X509_STRICT` flag is set. The fix removes only that flag. Certificate verification stays required (`CERT_REQUIRED`), and the client still verifies against the EKS CA. The final version adds the test that the only review comment asked for, and two reviewers approved it. Thus the oracle has no findings.

The approve datapoint also tests judgment. The change edits TLS settings, so it looks dangerous. An agent that blocks each change that looks dangerous fails this datapoint. An agent that blocks all PRs gets a perfect score on reject datapoints, and the approve datapoint finds this problem.

**Make sure that an approval is correct.** A human approval does not prove that the code is correct. Before you use an approved PR as an approve datapoint, examine the later commits to the same files. If a later commit fixes the same code, the approval was probably wrong. An earlier version of this dataset used [#4268](https://github.com/awslabs/mcp/pull/4268), an IAM policy guard that two reviewers approved. Codex blocked it: `iam:*` on `arn:aws:iam::*:role/*` was not blocked by the guard. Ten weeks later, [#4701](https://github.com/awslabs/mcp/pull/4701) closed this gap. The agent was correct, and the datapoint was wrong. For PR 4121, no later commit changes the code.

The candidate files in `datasets/awslabs-mcp-pr-review/candidates/` record the data from step 3 and the decisions from step 4. This cell shows a summary, and the high-severity finding of PR 4615 with its evidence.

In [ ]:
PR_DATASET = MODULE / "datasets" / "awslabs-mcp-pr-review"
for pr in (4615, 4121):
    cand = read_json(PR_DATASET / "candidates" / f"pr-{pr}.json")
    states = sorted({v["state"] for v in cand["review_verdicts"]})
    labels = {}
    for c in cand["comments"]:
        labels[c["classification"]] = labels.get(c["classification"], 0) + 1
    severities = [f["severity"] for f in cand["findings"]]
    risk = cand["change_risk"]
    print(f"PR {pr}: {len(cand['iterations'])} iterations, review states {states}")
    print(f"  comment labels: {labels}")
    print(f"  findings:       {severities or 'none'}")
    print(f"  risk:           structural {risk['risk_structural']}, classified {risk['risk_classified']}: {risk['risk_classified_rationale'][:160]}\n")

cand = read_json(PR_DATASET / "candidates" / "pr-4615.json")
high = next(f for f in cand["findings"] if f["severity"] == "high")
for key in ("statement", "severity_evidence", "severity_rationale"):
    print(f"{key}: {high[key]}\n")

### The oracle of a PR review task

`tests/oracle.json` is the correct answer. For each finding, make sure that:

- It is a defect, not a nit.
- It has a severity: `low`, `medium` or `high`. Only `high` findings block a merge.
- Its location touches a line that `change.patch` adds. The agent can find only what the change contains.

A location is a span of lines. An agent finding passes the location step if its line is in a span of the oracle finding. Then the judge decides if the two findings describe the same problem. A span that is too narrow is unfair: a correct finding on the next line gets no credit. Thus the high finding of the reject task has two spans: the function that strips the comments and runs the scan, and the definition of the statement-start pattern.

A `reference_only` finding is a real defect, but the agent does not have to find it. It counts in the overall coverage, but not in the required coverage. Also make sure that `expected_verdict` is `block` for the reject task and `approve` for the approve task.

In [ ]:
def added_lines(patch: str) -> dict[str, set[int]]:
    """Return the line numbers that a unified diff adds, for each file."""
    added, path, line_no = {}, None, None
    for line in patch.splitlines():
        if line.startswith("+++ "):
            path = line[6:] if line.startswith("+++ b/") else None
            added.setdefault(path, set())
        elif m := re.match(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@", line):
            line_no = int(m.group(1))
        elif path and line_no is not None and not line.startswith(("---", "\\")):
            if line.startswith("+"):
                added[path].add(line_no)
                line_no += 1
            elif line.startswith(" "):
                line_no += 1
    return added


for name in ("pr4615-reject", "pr4121-approve"):
    task_dir = TASKS[name].dir
    oracle = read_json(task_dir / "tests" / "oracle.json")
    expected = read_toml(task_dir / "task.toml")["metadata"]["harvest"]["expected_verdict"]
    changed = added_lines((task_dir / "environment" / "change.patch").read_text())
    print(f"{name}: expected_verdict={oracle['expected_verdict']}")
    assert oracle["expected_verdict"] == expected
    for f in oracle["findings"]:
        assert f["severity"] in ("low", "medium", "high")
        for loc in f["locations"]:
            assert set(range(loc["line_start"], loc["line_end"] + 1)) & changed.get(loc["path"], set()), f"Finding {f['id']} does not touch an added line."
        spans = ", ".join(f"{l['line_start']}-{l['line_end']}" if l["line_end"] != l["line_start"] else str(l["line_start"]) for l in f["locations"])
        print(f"  {f['severity']:<6} {f['locations'][0]['path'].split('/')[-1]}:{spans}{' (reference only)' if f['reference_only'] else ''}")
        print(f"         {f['statement'][:110]}...")
    assert (expected == "block") == any(f["severity"] == "high" for f in oracle["findings"]), "Only a task with a high finding can expect `block`."
    print()
print("oracle.json checks passed.")

### The rubric and the risk map

`rubric.md` tells the agent that labels the comments (step 4 above) how to set a severity. It starts from a standard review rubric. Then it adds the conventions of awslabs/mcp, from files such as `DESIGN_GUIDELINES.md` and `CONTRIBUTING.md`. These conventions are the rules of the team, which public benchmarks do not have.

Severity and risk are different. Severity tells how bad a finding is. Risk tells how dangerous the change is. Record the two values separately:

- **Structural risk.** `risk-map.toml` maps path prefixes to risk levels. For example, `src/iam-mcp-server/` is `high`. If a path is not in the map, the risk is `medium`.
- **Classified risk.** A model or a person sets the risk from the content of the change.

Use risk to decide if the agent can approve low-risk PRs without a human. The risk map does not include `src/eks-mcp-server/`, so the structural risk of PR 4121 is the default, `medium`. Its classified risk is `high`, because the change edits certificate verification.

In [ ]:
print((PR_DATASET / "rubric.md").read_text())
print((PR_DATASET / "risk-map.toml").read_text())

### The coding datapoints

A coding datapoint is a real bug: an issue report, the commit before the fix, and the fix that the team merged. The agent gets the issue report and the repository, and it must write the fix.

| Part | PR review datapoint | Coding datapoint |
|------|--------------------------|-----------------------|
| Start state | The base commit, with the change applied | The base commit (the parent of the fix), with no change |
| Instruction | The same for all tasks: review the change | An issue report, different for each task |
| Agent output | A list of findings | Changes to the code |
| Oracle | The defects that the reviewers found | The fix that the team merged |
| Verifier | An LLM judge matches the findings to the oracle | A reward stack: tests, a validation agent, a rubric judge and a taste judge |

Both bugs give the same symptom: a tool-level evaluator (for example, `ToolSelectionAccuracyEvaluator`) gives false negatives. The judge says that a correct tool call is wrong, because the extractor gives it the wrong context. The causes are different.

| | Root agent datapoint | Tool history datapoint |
|---|---|---|
| Issue / PR | [#370](https://github.com/strands-agents/evals/issues/370) / [#371](https://github.com/strands-agents/evals/pull/371) | [#337](https://github.com/strands-agents/evals/issues/337) / [#338](https://github.com/strands-agents/evals/pull/338) |
| Symptom | In a flat multi-agent trace, the judge thinks that the user asked only a part of the question | The judge says that an argument is hallucinated, but the value came from an earlier tool call |
| Cause | The root agent is the first agent in the span list, not the agent that started first | All tool calls in a trace get the same history, with no results of the earlier calls in that trace |
| Size of the merged fix | 1 function, 7 lines | 2 files, approximately 40 lines |
| Hidden tests | 6 fail-to-pass, 5 pass-to-pass | 7 fail-to-pass, 6 pass-to-pass |

The fix for the root agent is small, but the agent must find which function causes the symptom. The fix for the tool history is larger, and it has edge cases: parallel tool calls, calls with equal times, and spans that are not in time order.

The hidden tests are in two groups:

- **Fail to pass (f2p).** These tests fail on the base commit and pass with the merged fix. They show that the agent fixed the bug.
- **Pass to pass (p2p).** These tests pass on the base commit and must still pass. They show that the agent did not break something. The verifier also runs the full unit test suite of the repository at the base commit (`existing_suite`).

`[metadata]` in `task.toml` records the origin, the hidden tests, and a description of the cause and the fix. The agent does not see `task.toml`.

In [ ]:
for name in ("pr371-fix", "pr338-fix"):
    t = TASKS[name]
    meta = read_toml(t.dir / "task.toml")["metadata"]
    origin, verifier, story = meta["origin"], meta["verifier"], meta["narrative"]
    print(t.full)
    print(f"  origin: {origin['repo']} PR {origin['pr_numbers']}, issue {origin['issue_numbers']}, base {origin['base_commit'][:9]}")
    print(f"  tests:  {verifier['fail_to_pass']} fail-to-pass, {verifier['pass_to_pass']} pass-to-pass, in {verifier['hidden_tests']}")
    print(f"  cause:  {' '.join(story['description'].split())}")
    print(f"  fix:    {' '.join(story['solution'].split())}\n")
t = TASKS["pr371-fix"]   # the coding task of the next cells

### The hidden tests, the validation spec, the rubric and the taste judge

**The hidden tests** use only the public API. They check behavior, not the design of the fix. This cell prints the start of one hidden test file.

**The validation spec.** Each story has a description, a procedure and cases. The validation agent gets the stories and the agent's change. It does not get the oracle. The driver runs each test once for each case, so the validation agent cannot hard-code the answers. The procedure tells it the one mock that it can use: a stub for the judge model.

**The task rubric** holds requirements that are important but that a test cannot check well, or that are preferences. A single failed criterion does not fail the task: the score must be above 0.5.

**The taste judge** uses one global rubric for all tasks. It reads the conventions of the repository (for example `AGENTS.md` and `STYLE_GUIDE.md`) and compares the agent's change with the merged fix. It answers ten yes/no checks, for example "Is the change focused on the task, with no unrelated changes?" A check passes when a maintainer would accept the change on that point. There is no 1-5 scale: LLM judges that score on a scale tend to give 4 or 5 to everything, and a yes/no question is easier to check against a human.

`judge/models.toml` has the model, the temperature (0) and the turn budget of each LLM stage, and the thresholds of the gates.

In [ ]:
coding_tests = t.dir / "tests"
print(next((coding_tests / "verify").glob("*_verify.py")).read_text()[:2500])
print("\n==== validate/validation_spec.toml (start)\n" + (coding_tests / "validate" / "validation_spec.toml").read_text()[:2500])
print("\n==== judge/rubric.toml\n" + (coding_tests / "judge" / "rubric.toml").read_text())
print("==== judge/models.toml\n" + "\n".join(l for l in (coding_tests / "judge" / "models.toml").read_text().splitlines() if l and not l.startswith("#")))

### Leakage scan

Scan all files that the agent sees: `instruction.md` and `environment/`. Search for the PR and issue numbers, the reviewer names, text from the review comments, the merge commit, the name of the hidden test file, and lines that the fix adds. If the agent can see this data, the score measures search, not skill.

First, put a canary token in a temporary copy of the task, and make sure that the scan finds it. If the scan does not find it, the scan has a defect. The task files in the module do not change.

In [ ]:
def agent_visible_files(task_dir: Path) -> list[Path]:
    return [task_dir / "instruction.md", *sorted(p for p in (task_dir / "environment").rglob("*") if p.is_file())]


def scan(task_dir: Path, needles: list[str]) -> list[tuple[str, str]]:
    hits = []
    for path in agent_visible_files(task_dir):
        text = path.read_text(errors="ignore").lower()
        hits += [(str(path.relative_to(task_dir)), n) for n in needles if n and n.lower() in text]
    return hits


def needles_for(t) -> list[str]:
    meta = read_toml(t.dir / "task.toml")["metadata"]
    origin = meta["origin"]
    needles = [f"#{n}" for n in origin["pr_numbers"] + origin.get("issue_numbers", [])] + [f"pull/{n}" for n in origin["pr_numbers"]]
    if t.kind == "pr":       # the reviewer names and the start of each long review comment
        cand = read_json(PR_DATASET / "candidates" / f"pr-{origin['pr_numbers'][0]}.json")
        needles += sorted({v["author"] for v in cand["review_verdicts"]}) + [c["body"][:60] for c in cand["comments"] if len(c["body"]) >= 60]
    else:                    # the merge commit, the hidden test file and the long lines that the fix adds
        needles += [origin["merge_commit"][:12], Path(meta["verifier"]["hidden_tests"]).stem]
        needles += [l[1:].strip() for l in (t.dir / "solution" / "oracle.patch").read_text().splitlines()
                    if l.startswith("+") and not l.startswith("+++") and len(l.strip()) >= 40]
    return needles


CANARY = "CANARY-7f3a91"
for name, task in TASKS.items():
    with tempfile.TemporaryDirectory(prefix="leakage-canary-") as tmp:   # test the scan itself first
        copy = Path(tmp) / task.dir.name
        shutil.copytree(task.dir, copy)
        (copy / "environment" / "canary.txt").write_text(CANARY)
        assert scan(copy, [CANARY]), "The scan did not find the canary. The scan has a defect."
    needles = needles_for(task)
    hits = scan(task.dir, needles)
    print(f"{name:<15} {len(needles)} needles, " + ("no leaks." if not hits else f"LEAKS: {hits}"))

### The PR review grader

For each task, the agent writes a list of findings. `tests/score.py` compares the findings to the oracle in two steps. First, it compares the file and the line. Then an LLM judge (`JUDGE_MODEL_ID`) decides if the two findings describe the same problem.

An LLM judge can give different answers to the same question, most of all when the model does not accept a temperature. We saw Claude Haiku 5.5 say "no" to two identical findings in 1 of 4 calls. Thus `score.py` makes the decision stable in two ways:

- If the agent statement is the same as the oracle statement (case and whitespace do not matter), it is a match, with no judge call. The oracle agent submits the oracle statements, so the oracle control gives the same result each time.
- Otherwise, the judge asks the model `votes` times (3, in `tests/judge.toml`), and the majority decides. `verifier/judge.json` records each answer in `decisions`. A split vote (for example 2 yes and 1 no) shows a finding that is near the line between a match and no match. Examine it.

`tests/judge.toml` sets the temperature to 0. `score.py` sends it only if the model accepts it: some models reject the parameter, and then the judge uses the model default. `judge.json` records the model, the configured temperature, the temperature that was sent (`null` if none) and the number of calls.

`score.py` writes these signals to `reward.json`:

| Signal | Question |
|--------|----------|
| `coverage_required` | What fraction of the high (blocking) findings did the agent find? |
| `coverage_all` | What fraction of all oracle findings did the agent find? |
| `precision_adjudicated` | What fraction of the agent findings did the judge match to an oracle finding? Undefined if the agent submits no findings |
| `precision_strict` | What fraction of the agent findings have the same location as an oracle finding? No judge. Undefined if the agent submits no findings |
| `uncredited` | How many agent findings match no oracle finding? |
| `credited_<severity>`, `oracle_<severity>` | How many findings of each severity did the agent find, and how many are in the oracle? |

If `precision_strict` and `precision_adjudicated` are very different, the score depends much on the judge. Examine the judge. If the agent reports no findings, precision has no answer, and that is the correct result on an approve task, so a value of 0 would be wrong. Thus `score.py` leaves the precision keys out of `reward.json` and writes them as `null` in `verifier/reward_details.json`.

**The verdict.** The instruction tells the agent to request changes only when it finds a high-severity defect. Thus the verdict comes from the findings file: `block` if a finding has the severity `high`, and `approve` if not.

**Pass or fail.** Each task gets one result. This notebook uses a heuristic: pass if the verdict is correct, and for a reject task, `coverage_required` is 1.0. (An LLM judge with a rubric is another method: it decides the gray areas.) Over a dataset, these metrics follow:

| Metric | Question |
|--------|----------|
| **Accuracy** | What fraction of tasks did the agent pass? |
| **Block recall** | Of the PRs that must be blocked, what fraction did the agent block? |
| **Block precision** | Of the PRs that the agent blocked, what fraction must be blocked? |
| **False approve rate** | How frequently did the agent approve a bad PR? This error has the highest cost |
| **False block rate** | How frequently did the agent block a good PR? |

Obey these rules:

1. If the agent finds a problem that the reviewer did not find, record it as **uncredited**. Do not record it as wrong. Reviewers do not find all defects.
2. Keep each signal in the reward file. The signals show why a task failed.

In [ ]:
judge_cfg = read_toml(TASKS["pr4615-reject"].dir / "tests" / "judge.toml")["judge"]
print("PR review judge:", {k: judge_cfg[k] for k in ("provider", "model", "temperature", "votes")}, "| JUDGE_MODEL_ID:", JUDGE_MODEL_ID)

### The coding grader: the reward stack

A fix that passes the tests is not always a fix that a maintainer would merge. So the coding verifier grades the fix in the same way as [Senior SWE-Bench](https://senior-swe-bench.snorkel.ai/blog/2026-06-16-how-it-works): hidden tests, tests that a validation agent writes for the agent's change, a task rubric, a taste judge and patch bloat. A task is solved only if the fix is correct *and* tasteful.

| Mechanism | What it checks | Runs the code? | LLM? | Gate for a tasteful solve |
|-----------|----------------|----------------|------|---------------------------|
| **Hidden tests** | Fail-to-pass and pass-to-pass tests, and the base unit test suite | Yes | No | All pass |
| **Validation agent** | Tests that an agent writes from an expert *validation spec*, for the shape of the agent's change. A validation judge reviews the tests | Yes | Yes | All stories pass |
| **Task rubric judge** | Task-specific criteria that tests cannot check well, for example "the extractor and the trace use one rule to find the root" | No | Yes | Score > 0.5 |
| **Taste judge** | Five yes/no checks against the merged fix (minimality, approach, hygiene, fluency, craftsmanship), and five for fit with the codebase (style, patterns, libraries, abstraction, documentation) | No | Yes | At least 3 of 5 in each group |
| **Patch bloat** | Source lines of the agent's change divided by source lines of the merged fix | No | No | < 2x |
| **Agent-written tests** | Did the agent add a test that fails on the base and passes on its fix? A senior engineer does this without being told | Yes | No | Reported only |

Why have both hidden tests and a validation agent? A pre-written test can check only the interface that its author expected. The validation agent reads the agent's change and writes the test for the shape that the agent built. These datapoints are bug fixes, so the hidden tests already use only public interfaces. Here the validation agent tests at the system level: it runs the real evaluator with a stub model, and checks what the judge model would see.

`tests/test.sh` runs these stages after the agent stops. Their files are in `verifier/` of each trial record:

| Stage | Files | Output |
|-------|-------|--------|
| 0. Save the agent's change as a commit | | `agent.patch` |
| 1. Restore the base test files (the agent cannot change a test so that it passes), add the hidden tests, run all of them | `verify/` | `junit.xml`, `pytest.log` |
| 2. Run the agent-written tests on its fix and on the base | `lib/agent_tests.py` | `agent_tests.json` |
| 3. Validation agent and validation judge | `validate/validation_spec.toml`, `lib/validate.py` | `validation.json`, `validation_scripts/` |
| 4. Task rubric judge, taste judge, patch bloat | `judge/rubric.toml`, `judge/reference.patch`, `lib/judges.py` | `rubric.json`, `taste.json`, `bloat.json` |
| 5. Aggregate | `lib/aggregate.py` | `reward.json`, `reward_details.json` |

**How `reward` is formed.** `lib/aggregate.py` computes six gates (`checks` in `reward_details.json`): `verifiers`, `validation`, `rubric`, `patch_bloat`, `relative_taste` and `practice_alignment`. `correct` is 1 when the first two hold: the change works. `reward` (the same value as `tasteful_solve`) is 1 only when all six hold. The thresholds come from `[gates]` in `judge/models.toml`. `agent_tests_added` and `agent_tests_reproduce_bug` are reported, but they do not gate.

| Signal in `reward.json` | Question |
|--------|----------|
| `reward`, `tasteful_solve` | Is the change correct, and would a maintainer merge it? |
| `correct`, `verifiers_pass` | Does the change work? Do all pre-written tests pass? |
| `fail_to_pass`, `pass_to_pass`, `existing_suite` | What fraction of each group of pre-written tests passes? |
| `validation_score` | What fraction of the validation stories passes? |
| `rubric_score` | What fraction of the rubric criteria does the change fulfill? |
| `relative_taste`, `practice_alignment` | What fraction of the five taste checks passes in each group? |
| `patch_bloat` | How much larger is the change than the merged fix? |
| `agent_tests_added`, `agent_tests_reproduce_bug` | Did the agent add tests, and does one of them fail on the base? |

The resolved rate of an agent is the mean `correct`. The tasteful-solve rate is the mean `reward`. The difference tells you how often the agent writes a fix that works but that a reviewer would not merge. The taste gate is low on purpose: it removes clearly bad changes, and one changed answer of the judge does not change the result.

**Invalid trials.** If an LLM stage fails (for example, throttling), or the validation judge discards the validation tests, the trial is **invalid**: there is no `reward.json`, and `reward_details.json` says why. An invalid trial is not counted. An infrastructure failure is not a failure of the agent.

The verifier files are in `tests/` of each task. They stay in the module folder. After the agent stops, the runner copies `tests/` to a temporary folder, replaces the fixed paths (`/work/repo`, `/tests`, `/logs`, `/tmp`) with the trial's paths, and runs `tests/test.sh`. The files in `datasets/` do not change.

In [ ]:
show_tree(t.dir / "tests")
gates = read_toml(t.dir / "tests" / "judge" / "models.toml")["gates"]
print("\ngates:", gates, "| VERIFIER_MODEL_ID:", VERIFIER_MODEL_ID)

### The agent settings

The runner keeps its own copy of the model IDs (`MODELS`, and the judge models in `run_verifier`) and of the Bedrock hosts. This cell checks the agent model against `model_config.py`, and prints the Codex settings and flags of section 1. A `CHECK` means that you changed `model_config.py`, and the runner still uses its own value.

In [ ]:
same = runner.MODELS["codex"] == CODEX_MODEL_ID
print(f"{'OK   ' if same else 'CHECK'} agent model: runner {runner.MODELS['codex']}, model_config.py {CODEX_MODEL_ID}")
print("\ncodex-bedrock.toml:")
print("\n".join("  " + l for l in (MODULE / "codex-bedrock.toml").read_text().splitlines() if l and not l.startswith("#")))
print("\ncodex exec", " ".join(runner.CODEX_FLAGS), "<instruction>")

## 5. See What the Sandbox Lets the Agent Reach

The seal is only as good as the sandbox. This section looks at the sandbox profile, then runs the cheap stages of the test ladder and one extra probe.

### The agent's sandbox profile

The runner writes this profile for each trial. Reads of your home folder, `/Users`, `/Volumes` and `/tmp` are denied, except for the trial folder and the toolchain. Writes are allowed only in the trial folder. The network goes through `srt`'s proxy, which allows only the listed hosts.

In [ ]:
W = runner.WORK / "trials" / f"{t.full}__r1"
profile = runner.srt_settings(W, runner.AGENT_HOSTS, [W], [W], REAL_HOME)
print(mask(json.dumps(profile, indent=1)))

### Run the cheap stages of the test ladder

`run_all.sh` runs the test ladder of TESTER-GUIDE.md section 5. Three stages are cheap:

- **L0** starts the sandbox: `srt -- true`, then `echo sandbox-ok` inside it. If it prints `sandbox_apply: Operation not permitted`, Jupyter runs inside another sandbox.
- **L2** runs `scripts/l2_inner.sh` inside the agent's sandbox: both Bedrock hosts must answer, and GitHub, PyPI, npm, plain HTTP, IP addresses, IPv6, UDP, DNS and raw sockets must be blocked.
- **L6a** starts `node`, then asks Codex to reply "OK" inside the agent's sandbox. This checks that Codex can reach its model on Bedrock. If it cannot, the stage tries other settings and says which one works.

About 2 minutes, a few cents. `run_all.sh` writes each stage's log to `logs/run_all/<stage>.log`.

In [ ]:
local('STAGES="L0 L2 L6a" ./run_all.sh')

In [ ]:
def table(header: list[str], rows: list[list]) -> None:
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c).replace("|", "/") for c in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


l0 = LOCAL / "logs" / "run_all" / "L0.log"
print("L0:", l0.read_text().strip() if l0.exists() else "no log yet (run the cell above)")
gates = LOCAL / "logs" / "L2-gates.txt"
rows = []
for line in (gates.read_text().splitlines() if gates.exists() else []):
    if line.startswith("GATE|"):
        _, name, expected, observed, _detail = (line.split("|") + [""] * 5)[:5]
        if expected.startswith("reachable"):
            good = observed.startswith("reachable")
        elif expected == ">0":
            good = observed.strip().isdigit() and int(observed) > 0
        else:
            good = observed.startswith("blocked")
        rows.append(["OK" if good else "CHECK", name, expected, mask(observed)])
table(["", "L2 network gate", "expected", "observed"], rows) if rows else print("No L2 gates yet (run the cell above).")

### What the agent can and cannot read

This probe runs inside the agent's sandbox for the coding task, through the runner's `sandbox-run` command (the same command that stage L2 uses). It tries to read a hidden test and the merged fix in the module folder, to list your home folder and `~/.ssh`, and to write outside its trial folder. It reads nothing from these places: it prints only `readable`, `listed`, `written` or `denied`. Then it checks the git seal from the agent's point of view.

The probe script is written to `logs/`, which git ignores.

In [ ]:
hidden = next((t.dir / "tests" / "verify").glob("*_verify.py"))
oracle_patch = t.dir / "solution" / "oracle.patch"
probe = f"""#!/bin/bash
# Runs inside the agent's sandbox. One line per check: CHECK|what|expected|observed
c() {{ echo "CHECK|$1|$2|$3"; }}
r() {{ cat "$1" > /dev/null 2>&1 && echo readable || echo denied; }}
l() {{ ls "$1" > /dev/null 2>&1 && echo listed || echo denied; }}
c "read a hidden test in the module folder" denied "$(r {shlex.quote(str(hidden))})"
c "read the merged fix in the module folder" denied "$(r {shlex.quote(str(oracle_patch))})"
c "list your home folder" denied "$(l "$P1_REAL_HOME")"
c "list ~/.ssh" denied "$(l "$P1_REAL_HOME/.ssh")"
c "write a file in your home folder" denied "$(touch "$P1_REAL_HOME/.p1-nb-probe" 2>/dev/null && rm -f "$P1_REAL_HOME/.p1-nb-probe" && echo written || echo denied)"
c "write in its own trial folder" written "$(touch ../tmp/probe 2>/dev/null && echo written || echo denied)"
c "commits in the clone (git log --all)" 1 "$(git log --all --oneline 2>/dev/null | wc -l | tr -d ' ')"
c "git remotes in the clone" 0 "$(git remote 2>/dev/null | wc -l | tr -d ' ')"
c "hidden tests or oracle files in the trial folder" 0 "$(find .. -name '*_verify.py' -o -name oracle.patch -o -name reference.patch 2>/dev/null | wc -l | tr -d ' ')"
c "HTTPS to github.com" blocked "$(curl -fsS -o /dev/null --max-time 10 https://github.com/ 2>/dev/null && echo REACHED || echo blocked)"
"""
(LOCAL / "logs").mkdir(exist_ok=True)
(LOCAL / "logs" / "nb-probe.sh").write_text(probe)

out = local(f"{P1} sandbox-run pr371-fix --tag nbprobe --script logs/nb-probe.sh", check=False, capture=True)
rows = [line.split("|")[1:4] for line in out.splitlines() if line.startswith("CHECK|")]
table(["what the agent tries", "expected", "observed", ""], [[w, e, o, "OK" if o == e else "CHECK"] for w, e, o in rows]) if rows else None
for leftover in runner.WORK.glob("trials/*__nbprobe*"):   # the probe's trial folder is not needed after this
    shutil.rmtree(leftover, ignore_errors=True) if leftover.is_dir() else leftover.unlink()

A `CHECK` in the last column means the observed value is not the expected one. Do not run an agent until you know why: send the table with the feedback template at the end of TESTER-GUIDE.md.

### The sealed clones, from the host

The same seal, checked on the four sealed clones from outside the sandbox: one commit, no remote, no reflog, and no hidden test file.

In [ ]:
rows = []
for name, task in TASKS.items():
    repo = task.base / "repo"
    if not repo.is_dir():
        rows.append([name, "not prepared (run section 3)", "", "", "", ""])
        continue
    git = lambda *args: subprocess.run(["git", *args], cwd=repo, capture_output=True, text=True, env=SHELL_ENV).stdout
    commits, remotes = len(git("log", "--all", "--oneline").splitlines()), len(git("remote").split())
    reflog, hidden_files = len(git("reflog", "--all").splitlines()), len(list(repo.rglob("*_verify.py")))
    ok = commits == 1 and remotes == 0 and reflog == 0 and hidden_files == 0
    rows.append([name, "OK" if ok else "CHECK", commits, remotes, reflog, hidden_files])
table(["task", "seal", "commits", "remotes", "reflog entries", "hidden test files"], rows)

## 6. Run One PR-Review Trial and One Coding Trial

`p1 trial` does one trial:

1. Mint the two short-lived credentials of section 1 with `aws sts`.
2. Start the environment: a copy of the sealed clone and a fresh `uv` virtual environment.
3. Run Codex inside the agent's sandbox, with the flags of section 1.
4. After Codex stops, run the verifier inside its own sandbox.
5. Record the trial in `jobs/<job>/<trial>/`, then delete the trial folder.

The runner takes the lock `.bedrock.lock`, so only one trial runs at a time. A trial with the same job and trial name replaces the earlier record, so you can run these cells again.

### PR review: the approve task

Measured: about 52 s in total, about $0.13 for the agent.

In [ ]:
local(f"{P1} trial pr4121-approve --agent codex --job nb {FED}")

This cell reads the record: the timings of each phase, what the agent used, the reward, the verdict, the findings and the judge's votes.

In [ ]:
def seconds_between(start: str, end: str) -> float:
    parse = lambda s: datetime.fromisoformat(s.replace("Z", "+00:00"))
    return (parse(end) - parse(start)).total_seconds()


def trial_dir(job: str, name: str, tag: str = "r1") -> Path:
    return LOCAL / "jobs" / job / f"{TASKS[name].full}__{tag}"


def show_trial(job: str, name: str, tag: str = "r1") -> dict | None:
    """Print the timings, the agent's usage and the reward of one recorded trial. Return the reward."""
    d = trial_dir(job, name, tag)
    if not (d / "result.json").exists():
        print(f"No record at jobs/{job}/{d.name}. Run the trial cell first.")
        return None
    res = read_json(d / "result.json")
    env, agent, ver = res.get("environment_setup", {}), res.get("agent_result") or {}, res.get("verifier") or {}
    print(f"== jobs/{job}/{d.name}")
    print(f"  total {seconds_between(res['started_at'], res['finished_at']):.0f} s: environment {env.get('clone_s', 0) + env.get('venv_s', 0):.0f} s, "
          f"agent {agent.get('seconds', 0):.0f} s, verifier {ver.get('seconds', 0):.0f} s")
    if agent:
        print(f"  agent: exit {agent.get('exit')}, {agent.get('steps')} steps, {agent.get('tool_calls')} tool calls, "
              f"{agent.get('input_tokens')} input tokens ({agent.get('cached_input_tokens')} cached), {agent.get('output_tokens')} output tokens")
    if ver.get("stages"):
        print("  verifier stages (s):", ver["stages"])
    reward_file = d / "verifier" / "reward.json"
    if not reward_file.exists():
        details = d / "verifier" / "reward_details.json"
        why = read_json(details).get("invalid") if details.exists() else "no verifier output"
        print(f"  INVALID trial (no reward.json): {why}. An invalid trial is not a score of 0.")
        return None
    reward = read_json(reward_file)
    print("  reward:", {k: v for k, v in reward.items() if isinstance(v, (int, float))})
    return reward


reward = show_trial("nb", "pr4121-approve")
findings_file = trial_dir("nb", "pr4121-approve") / "artifacts" / "logs" / "artifacts" / "findings.json"
if reward is not None:
    findings = read_json(findings_file) if findings_file.exists() else []
    verdict = "block" if any(f.get("severity") == "high" for f in findings) else "approve"
    print(f"\n  verdict: {verdict} (expected: approve), {len(findings)} findings")
    for f in findings:
        print(f"    [{f.get('severity')}] {f.get('path', '').split('/')[-1]}:{f.get('line')} {f.get('statement', '')[:110]}")
    judge_file = trial_dir("nb", "pr4121-approve") / "verifier" / "judge.json"
    judge = read_json(judge_file) if judge_file.exists() else {}
    print(f"  judge: {judge.get('model')}, {judge.get('calls')} calls, {judge.get('exact_matches')} exact matches")
    for d in judge.get("decisions", []):
        print(f"    oracle finding {d['oracle_id']} vs agent finding {d['agent_index']}: votes {d['answers']}, accepted {d['accepted']}")

### Coding: the root agent bug

Measured: about 5 minutes in total (agent about 85 s, verifier about 3 minutes, of which the hidden and base tests are about 100 s), about $0.21 for the agent and $0.02 to $0.03 for the judges.

In [ ]:
local(f"{P1} trial pr371-fix --agent codex --job nb {FED}")

### Read a reward file

`verifier/reward.json` has only numbers: the signals of section 4. For a coding task, read it in this order:

1. `reward`: 1 is a tasteful solve. If it is 0, look at `correct`.
2. `correct`: if it is 1, the change works, and a taste, rubric or bloat gate failed. If it is 0, look at `fail_to_pass`, `pass_to_pass`, `existing_suite` and `validation_score`.
3. `verifier/reward_details.json` has the gates (`checks`), each failed test, each validation story, and the reason of each rubric criterion and each taste check.

No `reward.json` means an invalid trial: `reward_details.json` has the reason in `invalid`. This cell prints the gates, the signals, and the reasons for each failed check.

In [ ]:
SHOW = ["reward", "correct", "fail_to_pass", "pass_to_pass", "existing_suite", "validation_score",
        "rubric_score", "relative_taste", "practice_alignment", "patch_bloat"]
reward = show_trial("nb", "pr371-fix")
details_file = trial_dir("nb", "pr371-fix") / "verifier" / "reward_details.json"
if reward and details_file.exists():
    details = read_json(details_file)
    status = "TASTEFUL SOLVE" if reward.get("reward") == 1.0 else ("CORRECT, NOT TASTEFUL" if reward.get("correct") == 1.0 else "FAIL")
    print(f"\n  {status}")
    for gate, ok in details.get("checks", {}).items():
        print(f"    {'pass' if ok else 'FAIL'}  {gate}")
    print("  ", {k: reward.get(k) for k in SHOW})
    for line in details.get("failed_tests", [])[:5]:
        print(f"  failed test: {line}")
    for name, check in (details.get("taste", {}).get("dimensions") or {}).items():
        if isinstance(check, dict) and check.get("passed") is False:
            print(f"  taste check failed: {name}: {check.get('reason', '')[:140]}")
    for c in details.get("rubric", {}).get("criteria", []):
        if c.get("score") == 0:
            print(f"  rubric criterion failed: {c['id']}: {c.get('reason', '')[:140]}")
    print("  models:", details.get("models"))

### Open the agent's patch and its log

Each record has these files:

| File | Content |
|---|---|
| `result.json` | the task, the agent, the timings of each phase, the agent's tokens and steps |
| `verifier/reward.json`, `verifier/reward_details.json` | the reward, and every gate, failed test, validation story, rubric criterion and taste check, with the judges' reasons |
| `verifier/judge.json` | PR review: the judge model, the calls and each vote |
| `verifier/agent.patch` | coding: the agent's change as the verifier saw it |
| `verifier/failed_tests.txt`, `verifier/pytest.log` | coding: the tests that failed, and why |
| `verifier/validation.json`, `verifier/validation_scripts/` | coding: the tests that the validation agent wrote, their results, and the review of the validation judge |
| `verifier/rubric.json`, `verifier/taste.json` | coding: the result and the reason for each criterion and each taste check |
| `verifier/test.sh.out` | the verifier's output. The line `could not write config file .git/config` is expected: the sandbox denies it, and the runner sets the git identity another way |
| `verifier/render.diff` | the path substitutions in the verifier files |
| `agent/codex.txt` | Codex's JSON event stream. `Reconnecting... (Connection failed ...)` lines mean that it could not reach Bedrock |
| `agent/sessions/` | Codex's own session log, with every step |
| `artifacts/logs/artifacts/findings.json` | PR review: the agent's findings |

This cell prints the start of the agent's patch next to the merged fix, and the first commands that Codex ran.

In [ ]:
d = trial_dir("nb", "pr371-fix")
if (d / "verifier" / "agent.patch").exists():
    print(f"Record folder: {mask(str(d.relative_to(LOCAL)))}\n")
    print((d / "verifier" / "agent.patch").read_text()[:2000])
    print("\n-- merged fix --")
    print((TASKS["pr371-fix"].dir / "solution" / "oracle.patch").read_text()[:2000])
    print("\n-- the first commands of the agent --")
    commands = []
    for line in (d / "agent" / "codex.txt").read_text(errors="replace").splitlines():
        try:
            event = json.loads(line)
        except ValueError:
            continue
        item = event.get("item") or {}
        if event.get("type") == "item.completed" and item.get("type") == "command_execution":
            commands.append(str(item.get("command", "")).replace("\n", " ")[:150])
    print("\n".join(f"  $ {mask(c)}" for c in commands[:12]), f"\n  ({len(commands)} commands in total)")
else:
    print("No record yet. Run the coding trial first.")

**Questions to answer:**

1. The agent's patch is larger than the merged fix. Which part of it is a test that the agent wrote? Does `agent_tests_reproduce_bug` say that the test fails on the base commit?
2. Compare the agent seconds with the verifier seconds. Which phase would you make faster first, and how?

### Examine each failure

The score tells you that the agent failed. The patch and the log tell you why. The results change from run to run: the agent can find a defect in one run and miss it in the next. A failure that occurs in each trial is a weakness. A failure that occurs only sometimes is variance.

**Use a coding agent to read the records.** A log has many commands. A coding agent can read it faster than you can. For example:

```text
Read the records in jobs/nb/. For each PR review, compare artifacts/logs/artifacts/findings.json
with the task's tests/oracle.json, and read agent/codex.txt: what did the agent read, which inputs did
it try, and why did it decide that there was or was not a defect? For the coding trial, compare
verifier/agent.patch with the task's solution/oracle.patch, read verifier/failed_tests.txt, and if it
is correct but not tasteful, read verifier/taste.json and verifier/rubric.json and tell me if you
agree with the failed checks. Do not change files.
```

**Find the cause of a PR review failure:**

| Cause | Example | What to do |
|-------|---------|------------|
| **The agent misses a defect that it can find** | The agent read the guard, but it did not try the input that breaks it | Use the eval to improve the agent: change its prompt, its tools or its review procedure. Then run the eval again |
| **The agent does not have the context of the reviewer** | The reviewer knew how a different service uses the code. The repository does not show this | Give the PR agent more context, for example design documents, team rules, or the services that use the code |
| **The datapoint is wrong** | The agent blocked an approve task, and a later commit fixed the same defect | Correct the oracle, or select a different PR (section 4) |
| **The environment is broken** | A command failed because a tool is missing | Repair the task's environment file or `setup.sh`, then prepare the task again |
| **The grader did not give credit** | A correct finding is one line outside the span of the oracle finding | Make the span wider in `oracle.json` |

**Find the cause of a coding failure:**

| Cause | Example | What to do |
|-------|---------|------------|
| **The agent fixed the wrong thing** | The agent changed the evaluator prompt, not the extractor | Use the eval to improve the agent: its prompt, its tools or its procedure (for example, "write a failing test first") |
| **The fix is not complete** | `fail_to_pass` is 0.8: the fix fails on spans that are not in time order | Same as above. Also make sure that the instruction asks for this behavior |
| **The fix breaks something** | `existing_suite` is less than 1 | Same as above. The agent did not run the full test suite |
| **The hidden test checks the design, not the behavior** | A correct fix fails because a test expects one place in the history | Change the test so that a different correct fix passes. Then run the controls again |
| **The instruction does not ask for the behavior** | A test checks an edge case that the issue report does not tell | Add the behavior to the instruction, or remove the test |
| **The fix works, but it is not tasteful** | `correct` is 1, but `relative_taste` is 0.4: the agent patched the symptom in the evaluator, not the cause in the extractor | Use the eval to improve the agent. Read `taste.json` first: if you do not agree with the judge, change the judge prompt, and run the oracle again |
| **The validation tests are wrong** | The validation judge discarded the trial, or a story fails with an import error in the test code | Make the procedure in `validation_spec.toml` more precise. Then run the controls again |

## 7. Run the Controls

Before you trust a score, check the grader with two agents whose answer you know:

- **`nop`**, the empty agent, changes nothing. It must score 0.
- **`oracle`** submits the reference answer from `solution/`. It must score 1. If it does not, the grader has a defect.

For a coding task the rules are more exact:

- **Oracle:** a tasteful solve, `reward` 1.
- **Empty agent:** `fail_to_pass`, `correct` and `reward` are 0. `pass_to_pass` and `existing_suite` are 1. If a fail-to-pass test passes here, it does not test the bug. If a pass-to-pass test fails here, the test or the environment is wrong. The validation stories fail.

Measured on this Mac (one run each):

| Task | Run | f2p | p2p | existing | validation | rubric | taste checks (rel. / practice) | bloat | reward |
|------|-----|-----|-----|----------|------------|--------|-------------------------|-------|--------|
| pr371-fix | oracle | 1.0 | 1.0 | 1.0 | 2/2 stories | 1.0 | 5/5 / 5/5 | 1.0 | **1** |
| pr371-fix | empty | 0.0 | 1.0 | 1.0 | 0/2 | 0.5 | 0 / 0 | 0 | 0 |
| pr338-fix | oracle | 1.0 | 1.0 | 1.0 | 2/2 stories | 1.0 | 5/5 / 5/5 | 1.0 | **1** |
| pr338-fix | empty | 0.0 | 1.0 | 1.0 | 0/2 | 0.5 | 0 / 0 | 0 | 0 |

The empty agent gets some rubric credit, because an empty change fulfills the pass-to-pass criteria ("no interface change"). This is why the rubric does not gate a solve alone. A stricter calibration runs each control 3 times, and rejects a task unless all 3 results are correct (pass^3).

No model runs for the agent, so a control costs only the judge calls. On the reject PR-review task each control takes about 15 s (measured).

In [ ]:
for agent in ("oracle", "nop"):
    local(f"{P1} trial pr4615-reject --agent {agent} --job nb-ctl --tag {agent}")

expected = {"oracle": 1.0, "nop": 0.0}
for agent, value in expected.items():
    reward = show_trial("nb-ctl", "pr4615-reject", agent)
    if reward is not None:
        assert reward.get("coverage_all") == value, f"The {agent} control must have coverage_all {value}. The grader is broken."
        print(f"  OK: {agent} has coverage_all {value}\n")

On the **approve** task, the empty agent gets coverage 1.0, because there is no defect to find and an empty review approves. That is why the controls use the reject task.

The coding controls take about 4 minutes each, because the verifier runs the test suites and the judges. Set `RUN_CODING_CONTROLS = True` to run them on the coding task: the oracle must have `reward` 1, and the empty patch `reward` 0.

In [ ]:
RUN_CODING_CONTROLS = False  # optional: about 8 minutes, a few cents of judge calls

if RUN_CODING_CONTROLS:
    for agent in ("oracle", "nop"):
        local(f"{P1} trial pr371-fix --agent {agent} --job nb-ctl --tag {agent}")
    for agent, value in {"oracle": 1.0, "nop": 0.0}.items():
        reward = show_trial("nb-ctl", "pr371-fix", agent)
        if reward is not None:
            assert reward.get("reward") == value, f"The {agent} control must have reward {value}. See verifier/reward_details.json."
            if agent == "nop":
                assert reward.get("fail_to_pass") == 0.0, "A fail-to-pass test passes without a fix: it does not test the bug."
                assert reward.get("pass_to_pass") == 1.0 and reward.get("existing_suite") == 1.0, "A test fails on the base."
            print(f"  OK: {agent} has reward {value}\n")
else:
    print("Skipped. Set RUN_CODING_CONTROLS = True to run the coding controls.")

## 8. The Set of 12 Trials, and How to Read It

One trial per task shows that the pipeline works. It does not rank agents. The measured set is 4 tasks x 3 Codex trials, one at a time: **37.7 minutes, about $2.60** (agent $2.46 and judges about $0.15, estimate). The command is the one in TESTER-GUIDE.md:

```bash
p1 job --tasks pr371-fix,pr338-fix,pr4121-approve,pr4615-reject --n 3 --agent codex --job nb-full --federated scripts/agent-policy.json
```

`./run_all.sh` with `STAGES=L7` runs the same set and then the oracle and empty controls on all four tasks (about 54 minutes). You can also run either in a plain terminal, after `source env.sh`.

`RUN_SLICE` runs a slice of 3 trials, one per task type, in about 7 minutes (about $0.55). `RUN_FULL_SET` runs the 12. Both are off.

In [ ]:
RUN_SLICE = False     # 3 trials: approve, reject, one coding task. About 7 minutes, about $0.55
RUN_FULL_SET = False  # 12 trials: 4 tasks x 3. About 38 minutes, about $2.60

if RUN_SLICE:
    local(f"{P1} job --tasks pr4121-approve,pr4615-reject,pr371-fix --n 1 --agent codex --job nb-slice {FED}")
if RUN_FULL_SET:
    local(f"{P1} job --tasks {','.join(TASK_NAMES)} --n 3 --agent codex --job nb-full {FED}")
if not (RUN_SLICE or RUN_FULL_SET):
    print("Skipped. Set RUN_SLICE or RUN_FULL_SET to True to run more trials.")

### Read the results

This cell reads every Codex job in `jobs/` that this notebook or `run_all.sh` made, one row per trial. A trial is **valid** only if the agent exited 0, used input tokens and the verifier wrote `reward.json`. The runner stops a job at the first trial that is not valid.

In [ ]:
def trial_rows(job_dir: Path) -> list[list]:
    rows = []
    for result_file in sorted(job_dir.glob("*/result.json")):
        d, res = result_file.parent, read_json(result_file)
        agent = res.get("agent_result") or {}
        name = res["task_name"].split("__")[-1]
        reward = read_json(d / "verifier" / "reward.json") if (d / "verifier" / "reward.json").exists() else None
        valid = agent.get("exit") == 0 and (agent.get("input_tokens") or 0) > 0 and reward is not None
        if not valid:
            score = "INVALID"
        elif "coverage_all" in reward:
            findings_file = d / "artifacts" / "logs" / "artifacts" / "findings.json"
            high = any(f.get("severity") == "high" for f in (read_json(findings_file) if findings_file.exists() else []))
            passed = (not high) if name.endswith("approve") else (high and reward.get("coverage_required") == 1.0)
            score = f"{'pass' if passed else 'fail'} (coverage_required {reward.get('coverage_required')})"
        else:
            score = f"reward {reward.get('reward')}, correct {reward.get('correct')}, taste {reward.get('relative_taste')}"
        rows.append([job_dir.name, d.name.split("__")[-2] if "__" in d.name else name, d.name.split("__")[-1],
                     score, agent.get("seconds"), f"{seconds_between(res['started_at'], res['finished_at']):.0f}"])
    return rows


rows = []
for job in ("nb", "nb-slice", "nb-full", "L6", "L7"):
    if (LOCAL / "jobs" / job).is_dir():
        rows += trial_rows(LOCAL / "jobs" / job)
table(["job", "task", "trial", "result", "agent s", "total s"], rows) if rows else print("No Codex trials recorded yet.")

### The PR review metrics

This cell applies the pass rule of section 4 to every valid PR-review trial in the same jobs, and computes the dataset metrics.

In [ ]:
reviews = []
for job in ("nb", "nb-slice", "nb-full", "L6", "L7"):
    for result_file in sorted((LOCAL / "jobs" / job).glob("awslabs__*/result.json")):
        d = result_file.parent
        agent = read_json(result_file).get("agent_result") or {}
        if not ((d / "verifier" / "reward.json").exists() and agent.get("exit") == 0 and (agent.get("input_tokens") or 0) > 0):
            continue
        task = TASKS[d.name.split("__")[-2]]
        expected = read_toml(task.dir / "task.toml")["metadata"]["harvest"]["expected_verdict"]
        findings_file = d / "artifacts" / "logs" / "artifacts" / "findings.json"
        findings = read_json(findings_file) if findings_file.exists() else []
        predicted = "block" if any(f.get("severity") == "high" for f in findings) else "approve"
        reward = read_json(d / "verifier" / "reward.json")
        reviews.append({"expected": expected, "predicted": predicted,
                        "passed": predicted == expected and (expected == "approve" or reward.get("coverage_required") == 1.0)})

rate = lambda part, whole: f"{len(part)}/{len(whole)}" if whole else "n/a"
should_block = [r for r in reviews if r["expected"] == "block"]
did_block = [r for r in reviews if r["predicted"] == "block"]
clean = [r for r in reviews if r["expected"] == "approve"]
print("reviews:           ", len(reviews))
print("accuracy:          ", rate([r for r in reviews if r["passed"]], reviews))
print("block recall:      ", rate([r for r in should_block if r["predicted"] == "block"], should_block))
print("block precision:   ", rate([r for r in did_block if r["expected"] == "block"], did_block))
print("false approve rate:", rate([r for r in should_block if r["predicted"] == "approve"], should_block))
print("false block rate:  ", rate([r for r in clean if r["predicted"] == "block"], clean))

**How to read these numbers.** The measured set gave reward 1 in 6 of 6 coding trials, pass on the approve task in 3 of 3, and fail on the reject task in 3 of 3. Keep these limits in mind:

- **The judges are noisy.** For the same patch, the taste judge's score moves with a standard deviation of about 0.19 per call (0.4 to 1.0 over 27 repeats). About 5 to 7% of trials on a pr338-style patch get reward 0 from the taste gate alone. With 3 trials per task, the chance of at least one such 0 is about 20%. One 0 is not evidence that an agent is worse.
- **The reject task is hard.** Codex found its defect in 12% of 185 earlier trials. With that rate, 0 of 3 has a probability of about 0.67. About 85 trials per design are needed to tell 12% from 30%.
- **n=3 per task shows only large differences.** A coding reward that is 1 almost every time cannot show that one setup is worse. With two tasks and one trial each, a rate can be only 0, 0.5 or 1.
- **Risk levels.** Before an agent approves low-risk PRs without a human, examine the low-risk results only, with sufficient low-risk datapoints. This dataset has no low-risk datapoints, so it cannot tell you that.

## 9. Clean Up

**Nothing exists in AWS.** The runner made only `sts` calls (temporary credentials, which expire on their own after 1 and 2 hours) and Bedrock invocations. There is no resource to delete. If model invocation logging is on in your account, these calls are logged there like any other Bedrock call.

On your Mac, the trial data is in two places, and the toolchain is in this folder:

| Path | Content |
|---|---|
| `~/.coding-assistant-local/<id>/work` | sealed clones, trial folders and verifier copies (can grow to several GB) |
| `jobs/`, `logs/`, `.bedrock.lock` in this folder | the records and logs |
| `node/`, `npm/`, `npm-cache/`, `uv-cache/`, `uv-python/`, `home/`, `bin/` in this folder | the toolchain and caches, and Jupyter's own packages if you started it with the command in the first cell |

This cell shows the sizes. Set `DELETE_TRIAL_DATA = True` to delete the first two rows. The toolchain is still in use by this kernel, so delete it from a terminal after you stop Jupyter, with the command that the cell prints. Git ignores all these paths, so `git status` stays clean.

In [ ]:
DELETE_TRIAL_DATA = False  # True deletes the work folder, jobs/, logs/ and the lock

def size(path: Path) -> str:
    if not path.exists():
        return "not present"
    ok, out = run(["du", "-sh", str(path)])
    return out.split()[0] if ok and out else "?"

TRIAL_DATA = [P1_WORK, LOCAL / "jobs", LOCAL / "logs", LOCAL / ".bedrock.lock"]
TOOLCHAIN = ["node", "npm", "npm-cache", "uv-cache", "uv-python", "home", "bin"]
for path in TRIAL_DATA + [LOCAL / name for name in TOOLCHAIN]:
    print(f"{size(path):>12}  {mask(str(path))}")

if DELETE_TRIAL_DATA:
    for path in TRIAL_DATA:
        shutil.rmtree(path, ignore_errors=True)
    for empty in (P1_WORK.parent, P1_WORK.parent.parent, Path("/tmp/claude")):   # empty parents of the work folder; srt's shared temp folder
        try:
            empty.rmdir()
        except OSError:
            pass
    print("\nDeleted the trial data.")

print("\nAfter you stop Jupyter, delete the toolchain from this folder in a terminal:")
print(f"  rm -rf {' '.join(TOOLCHAIN)}")

## Summary

- Your git history contains your eval data. Roll back to the parent commit, seal the environment, and use the PR review or the merged patch as the oracle.
- A task directory keeps the agent's files (`instruction.md`, `environment/`) apart from the verifier's files (`tests/`, `solution/`).
- Each trial here gets a sealed git clone, a `uv` environment and an `srt` sandbox that reaches only Amazon Bedrock. Start Jupyter (or the runner) from a plain terminal, and keep the checkout under your home folder.
- A review passes on the verdict and the required coverage. A fix must work *and* be one that a maintainer would merge: `reward` needs all six gates.
- The judges are a different model from the agent, and they vote or answer yes/no checks, so one changed answer does not change the result.
- Check the sandbox before you trust a score: the L2 network gates, the read probe and the seal. Check the grader with the controls: the empty agent scores 0, the oracle scores 1.
- One trial per task shows that the pipeline works. To rank agents, run more trials, and expect judge noise.

**Next:** run the whole test ladder with `./run_all.sh` in a plain terminal ([TESTER-GUIDE.md](TESTER-GUIDE.md), section 5), and send your results with the feedback template at the end of the guide. To make datapoints from your own repository, use [sample-code-eval-harvester](https://github.com/aws-samples/sample-code-eval-harvester).